[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn.functional as F

/Users/renyumeng/文稿/1.code/TorchCode/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
from torch_judge import hint
hint('conv2d')


💡 Hint for 2D Convolution:
   Extract patches using unfold or nested loops. For each output position, sum(patch * kernel). Support stride and padding (zero-pad with F.pad).



In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x:torch.Tensor, weight:torch.Tensor, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
    if padding>0:
        x = F.pad(x,[padding]*4)
    B,C_in,H,W = x.shape
    C_out,C_in,kH,kW = weight.shape
    H_out = (H-kH)//stride + 1
    W_out = (W-kW)//stride + 1
    # shape (B,C_in,H_out,W_out,kH,kW)
    patches = x.unfold(2,kH,stride).unfold(3,kW,stride)
    
    out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
    if bias is not None:
        out = out+bias.view(1,-1,1,1)
    
    return out

In [4]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.7ms)
  ✅ [2/5] Matches F.conv2d (24.4ms)
  ✅ [3/5] With padding (8.2ms)
  ✅ [4/5] With stride (3.3ms)
  ✅ [5/5] Gradient flow (83.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (123.9ms total)
  Progress saved. Run status() to see your dashboard.

